# Session 7: Structured Prompting with LangChain
## Chain-of-Thought | ReAct | Tree-of-Thought | Graph-of-Thought

> **BIA® School of Technology & AI — Generative AI & Agentic AI Development**

In Session 6 we learned *what* these techniques are. Today we **implement** them in Python using **LangChain** — the industry-standard framework for building LLM applications.

**What you'll learn:**
- LangChain Expression Language (LCEL) — the `prompt | model | parser` pattern
- Structured output with Pydantic models
- Tool-calling agents with LangGraph
- When (and when not) to use each prompting pattern

---

## Setup

Run these cells first. Install the packages if needed, then configure your API key.

In [ ]:
# Uncomment to install (run once)
# !pip install langchain-openai langchain-core langgraph pydantic

import os
import json
import math
from pydantic import BaseModel, Field

# ── LangChain core imports ──────────────────────────────────────────
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# ── Set your API key ────────────────────────────────────────────────
# Option 1: Set environment variable  (recommended)
#   export OPENAI_API_KEY="sk-..."
# Option 2: Pass directly (for quick testing only)
#   os.environ["OPENAI_API_KEY"] = "sk-YOUR-KEY-HERE"

# ── Initialize the model ────────────────────────────────────────────
# We use ChatOpenAI from langchain-openai (not the raw openai SDK).
# This gives us .invoke(), LCEL pipes, structured output, and tool binding.

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    max_tokens=1500,
)

# Quick test
response = llm.invoke("Say 'LangChain is ready!' in one sentence.")
print(response.content)

---
## LangChain Key Concepts (Quick Reference)

Before we dive in, here are the three LangChain patterns we'll use throughout:

| Pattern | Code | What it does |
|---------|------|--------------|
| **LCEL Chain** | `prompt \| llm \| parser` | Pipe data through prompt → model → parser |
| **Structured Output** | `llm.with_structured_output(MyModel)` | Force the model to return a Pydantic object |
| **Tool Calling** | `llm.bind_tools([my_tool])` | Let the model call Python functions |

**LCEL** = LangChain Expression Language. The `|` (pipe) operator connects components left-to-right.

---

## 1. Chain-of-Thought in Practice

Session 6 showed you `"Think step by step"`. That's **Level 1** (Zero-Shot CoT).

Today we go to **Level 2** (Structured CoT) and **Level 3** (Programmatic CoT with Pydantic).

| Level | What | How |
|-------|------|-----|
| 1. Zero-Shot CoT | "Think step by step" | One line added to prompt |
| 2. Structured CoT | Force specific sections | LCEL chain with section-based system prompt |
| 3. Programmatic CoT | Validated Pydantic output | `with_structured_output()` returns typed objects |

### 1.1 Structured CoT with LCEL

We build our first **LCEL chain**: `ChatPromptTemplate` → `ChatOpenAI` → `StrOutputParser`

The `|` pipe operator passes each component's output to the next.

In [ ]:
# ── Build the chain using LCEL ───────────────────────────────────────

# Step 1: Define the prompt template
structured_cot_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an analytical problem solver.

For every problem, follow this EXACT structure:

## Understanding
Restate the problem. What is given? What is asked?

## Approach
List 2 possible approaches. Select the best one.

## Step-by-Step Solution
Number each step. Show your work.

## Verification
Check your answer using a different method.

## Final Answer
One clear sentence."""),
    ("human", "{problem}")
])

# Step 2: Create the LCEL chain
#   prompt → model → parser
#   The pipe (|) operator connects them left-to-right
cot_chain = structured_cot_prompt | llm | StrOutputParser()

# Step 3: Run it with .invoke()
result = cot_chain.invoke({
    "problem": """A store offers 25% off all items. Loyalty members get an extra 10% off
the discounted price. Sales tax is 8%. If a loyalty member buys
an item originally priced at $200, what do they pay?"""
})

print(result)

**What just happened:**
1. `ChatPromptTemplate.from_messages()` created a reusable prompt with a `{problem}` placeholder
2. The `|` operator piped: prompt → model → string parser
3. `.invoke({"problem": ...})` ran the whole chain and returned plain text

**Why this is better than raw API calls:**
- The prompt is **reusable** — call `.invoke()` with different problems
- The chain is **composable** — swap the model or parser without changing anything else
- `StrOutputParser()` extracts just the text content (no `.choices[0].message.content` boilerplate)

### 1.2 Programmatic CoT — Pydantic Structured Output

For agentic workflows, we need **machine-readable** reasoning. LangChain's `with_structured_output()` 
forces the model to return a validated Pydantic object — no manual JSON parsing needed.

In [ ]:
# ── Define the output schema with Pydantic ──────────────────────────
from typing import List

class ReasoningStep(BaseModel):
    """A single step in the chain of thought."""
    step: int = Field(description="Step number (1, 2, 3...)")
    description: str = Field(description="What this step does")
    operation: str = Field(description="The math expression")
    result: float = Field(description="Numerical result of this step")

class CoTSolution(BaseModel):
    """Complete chain-of-thought solution with verification."""
    steps: List[ReasoningStep] = Field(description="Ordered reasoning steps")
    verification: str = Field(description="How you checked the answer")
    final_answer: float = Field(description="The final numerical answer")
    unit: str = Field(description="Unit of the answer (dollars, km, etc.)")

# ── Create a structured output chain ────────────────────────────────
# with_structured_output() binds the Pydantic schema to the model.
# The model will return a CoTSolution object (not raw text).

structured_llm = llm.with_structured_output(CoTSolution)

cot_structured_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a calculation agent. Break every math problem into "
               "individual steps. Execute each step and verify the final answer."),
    ("human", "{problem}")
])

# Chain: prompt → structured model (returns Pydantic object directly)
cot_json_chain = cot_structured_prompt | structured_llm

In [ ]:
# ── Test it ──────────────────────────────────────────────────────────
result = cot_json_chain.invoke({
    "problem": "A car travels at 65 mph for 3.5 hours. How far does it go?"
})

# result is a Pydantic object — access fields with dot notation
print(f"Answer: {result.final_answer} {result.unit}")
print(f"Steps:  {len(result.steps)}")
for step in result.steps:
    print(f"  Step {step.step}: {step.description} → {step.operation} = {step.result}")
print(f"Verification: {result.verification}")

# Validate step numbering automatically
assert all(s.step == i + 1 for i, s in enumerate(result.steps)), "Step numbering error!"
print("\n✓ Validation passed")

In [ ]:
# ── Try a harder one ─────────────────────────────────────────────────
result = cot_json_chain.invoke({
    "problem": """A bakery makes 240 cupcakes. They sell 35% in the morning and 
40% of the remaining in the afternoon. How many are left?"""
})

print(f"Answer: {result.final_answer} {result.unit}")
for step in result.steps:
    print(f"  Step {step.step}: {step.description} = {step.result}")

**Key takeaway:** `with_structured_output()` is the modern LangChain way to get machine-readable output. Compared to raw `response_format={"type": "json_object"}`:
- You define a **Pydantic schema** — fields are typed and validated automatically
- No `json.loads()` needed — you get a Python object with dot notation
- If the model returns invalid data, Pydantic raises a clear error

---

## 2. ReAct: Building an Agent Loop

ReAct (Reason + Act) is the **backbone of modern AI agents**. The model alternates between:

```
Thought  →  Action  →  Observation  →  Thought  →  ...  →  Final Answer
```

We'll build it **two ways**:
1. **From scratch** (text parsing) — to understand how it works
2. **With LangGraph** (`create_react_agent`) — the production approach

### 2.1 Define Tools with LangChain's `@tool` Decorator

In LangChain, tools are Python functions decorated with `@tool`. The decorator automatically 
extracts the function name, docstring, and type hints for the model to use.

In [ ]:
import re
from langchain_core.tools import tool

@tool
def search_web(query: str) -> str:
    """Search the web for factual information. Use this to look up data like
    population, GDP, or other statistics."""
    data = {
        "population of france": "The population of France is approximately 68.4 million (2024).",
        "gdp of france": "France's GDP is approximately $3.05 trillion (2024).",
        "population of germany": "The population of Germany is approximately 84.5 million (2024).",
        "gdp of germany": "Germany's GDP is approximately $4.07 trillion (2024).",
    }
    for key, value in data.items():
        if key in query.lower():
            return value
    return f"No results found for: {query}"


@tool
def calculate(expression: str) -> str:
    """Evaluate a math expression. Input should be a valid Python math expression
    like '68400000 / 3050000000000 * 100'."""
    try:
        result = eval(expression, {"__builtins__": {}},
                      {"abs": abs, "round": round, "pow": pow, "sqrt": math.sqrt})
        return str(result)
    except Exception as e:
        return f"Error: {e}"


# LangChain tools have auto-generated metadata
tools = [search_web, calculate]

for t in tools:
    print(f"Tool: {t.name}")
    print(f"  Description: {t.description}")
    print(f"  Args: {t.args}")
    print()

### 2.2 ReAct from Scratch (Text Parsing)

First, let's build the agent loop manually so you understand what's happening inside. 
We use LangChain's `ChatPromptTemplate` for the prompt but handle the loop ourselves.

In [ ]:
# ── ReAct from scratch using LangChain messages ─────────────────────

REACT_SYSTEM = """You are a helpful assistant that reasons step-by-step and uses tools.

## Available Tools
- search_web("query"): Search the web for information
- calculate("expression"): Evaluate a math expression

## Format
For each step, respond with:

Thought: [your reasoning]
Action: [tool_name("input")]
<PAUSE>

After receiving observations, continue or give the final answer:

Thought: [reasoning]
Final Answer: [your complete answer]

## Rules
- Always start with a Thought
- Use ONE tool per step
- Never guess — use tools to get facts
- Maximum 5 tool calls
"""

# Map tool names to their functions for dispatching
TOOL_MAP = {t.name: t for t in tools}

def react_agent_scratch(question: str, max_steps: int = 5) -> str:
    """ReAct agent built from scratch (for learning)."""
    messages = [
        SystemMessage(content=REACT_SYSTEM),
        HumanMessage(content=question),
    ]

    for step in range(max_steps):
        # Call the model via LangChain
        response = llm.invoke(messages)
        text = response.content

        print(f"\n--- Step {step + 1} ---")
        print(text)

        # Check for final answer
        if "Final Answer:" in text:
            return text.split("Final Answer:")[-1].strip()

        # Parse the action with regex
        action_match = re.search(r'Action:\s*(\w+)\("([^"]*)"\)', text)
        if not action_match:
            messages.append(AIMessage(content=text))
            messages.append(HumanMessage(content="Please use a tool or provide a Final Answer."))
            continue

        tool_name = action_match.group(1)
        tool_input = action_match.group(2)

        # Execute the tool
        if tool_name in TOOL_MAP:
            observation = TOOL_MAP[tool_name].invoke(tool_input)
        else:
            observation = f"Error: Unknown tool '{tool_name}'"

        print(f"\n>> Observation: {observation}")

        # Add to conversation
        messages.append(AIMessage(content=text))
        messages.append(HumanMessage(content=f"Observation: {observation}"))

    return "Max steps reached."

### 2.3 Test the From-Scratch Agent

In [ ]:
# Test the from-scratch agent
answer = react_agent_scratch("What is the GDP per capita of France? Show the calculation.")
print(f"\n{'='*50}")
print(f"FINAL ANSWER: {answer}")

In [ ]:
# Try a multi-step comparison
answer = react_agent_scratch("Which country has higher GDP per capita: France or Germany?")
print(f"\n{'='*50}")
print(f"FINAL ANSWER: {answer}")

### 2.4 Comparison: How does tool calling work?

Before we use LangGraph, let's see what `bind_tools()` does under the hood. 
This is the mechanism that replaces text parsing with structured JSON tool calls.

In [ ]:
# ── bind_tools() demo ────────────────────────────────────────────────
# bind_tools() tells the model about available tools.
# The model returns structured tool_calls instead of text.

llm_with_tools = llm.bind_tools(tools)

# Ask a question that requires a tool
response = llm_with_tools.invoke("What is the population of France?")

# The model returns tool_calls — structured JSON, no regex needed!
print("Content:", response.content)
print("Tool calls:", response.tool_calls)

if response.tool_calls:
    tc = response.tool_calls[0]
    print(f"\n  Tool: {tc['name']}")
    print(f"  Args: {tc['args']}")
    
    # Execute the tool
    result = TOOL_MAP[tc["name"]].invoke(tc["args"])
    print(f"  Result: {result}")

### 2.5 ReAct with LangGraph (Production Way)

LangGraph wraps the entire Thought → Action → Observation loop into `create_react_agent()`. 
This is what you use in production — one line replaces the entire agent loop.

In [ ]:
# ── LangGraph's create_react_agent (Production Way) ─────────────────
# LangGraph wraps the entire Thought → Action → Observation loop
# into a single function. This is what you use in production.

from langgraph.prebuilt import create_react_agent

# Create the agent — one line!
# It handles: tool binding, the agent loop, tool execution, and stopping.
agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt="You are a helpful research assistant. Think step by step.",
)

# Run the agent
result = agent.invoke({
    "messages": [{"role": "user", "content": "What is the GDP per capita of France?"}]
})

# Print the conversation
for msg in result["messages"]:
    role = msg.__class__.__name__
    if hasattr(msg, "content") and msg.content:
        print(f"[{role}] {msg.content[:200]}")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  → Calling: {tc['name']}({tc['args']})")

In [ ]:
# Try the multi-step comparison
result = agent.invoke({
    "messages": [{"role": "user", "content": "Which country has higher GDP per capita: France or Germany?"}]
})

# Get just the final answer
final_message = result["messages"][-1]
print("FINAL ANSWER:")
print(final_message.content)

**From-Scratch vs LangGraph:**

| | From Scratch (2.2) | LangGraph (2.5) |
|--|--|--|
| Tool dispatch | Regex parsing | Structured `tool_calls` |
| Agent loop | Manual `for` loop | Handled by framework |
| Error handling | DIY | Built-in retries |
| Lines of code | ~40 | ~5 |
| Use for | **Learning** | **Production** |

**The key insight:** LangGraph's `create_react_agent` does exactly what our from-scratch loop does — 
it just handles the plumbing so you can focus on defining tools and prompts.

---

## 3. Tree-of-Thought: Multi-Path Reasoning

CoT follows **one** path. But some problems have multiple valid approaches.

Tree-of-Thought explores multiple paths and picks the best:

```
            Problem
           /   |   \
      Path A  Path B  Path C
        ↓       ↓       ↓
    (weak)  (best!)  (okay)
              ↓
          Solution
```

We'll use LCEL chains with Pydantic structured output for each stage.

In [ ]:
# ── Tree-of-Thought with Pydantic schemas ───────────────────────────

class Approach(BaseModel):
    id: int
    strategy: str = Field(description="2-sentence description of the approach")
    first_step: str = Field(description="The first concrete step to take")

class ApproachList(BaseModel):
    approaches: List[Approach]

class Evaluation(BaseModel):
    id: int
    feasibility: int = Field(ge=1, le=5)
    efficiency: int = Field(ge=1, le=5)
    correctness: int = Field(ge=1, le=5)
    total: int = Field(description="Sum of the three scores")

class EvaluationResult(BaseModel):
    evaluations: List[Evaluation]
    best_approach_id: int


def tree_of_thought(problem: str, n_paths: int = 3) -> str:
    """Tree-of-Thought: Generate approaches, evaluate, pursue the best."""

    # ── Step 1: Generate approaches (higher temperature for diversity) ──
    print("Step 1: Generating approaches...")
    creative_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
    
    gen_prompt = ChatPromptTemplate.from_messages([
        ("system", "You propose multiple problem-solving strategies."),
        ("human", "Propose {n_paths} distinctly different approaches to this problem:\n\n{problem}")
    ])
    gen_chain = gen_prompt | creative_llm.with_structured_output(ApproachList)
    
    approaches = gen_chain.invoke({"n_paths": n_paths, "problem": problem})
    
    for a in approaches.approaches:
        print(f"  Approach {a.id}: {a.strategy[:80]}...")

    # ── Step 2: Evaluate each approach (low temperature for consistency) ──
    print("\nStep 2: Evaluating approaches...")
    eval_prompt = ChatPromptTemplate.from_messages([
        ("system", "You evaluate problem-solving approaches objectively."),
        ("human", "Rate each approach to this problem on feasibility, efficiency, "
                  "and correctness (1-5 each).\n\nProblem: {problem}\n\n"
                  "Approaches:\n{approaches_text}")
    ])
    eval_chain = eval_prompt | llm.with_structured_output(EvaluationResult)
    
    approaches_text = "\n".join(
        f"{a.id}. {a.strategy}" for a in approaches.approaches
    )
    evaluation = eval_chain.invoke({
        "problem": problem,
        "approaches_text": approaches_text,
    })
    
    for e in evaluation.evaluations:
        print(f"  Approach {e.id}: total={e.total}")
    print(f"  Best: Approach {evaluation.best_approach_id}")

    # ── Step 3: Execute the best approach ──
    print("\nStep 3: Executing best approach...")
    best = next(a for a in approaches.approaches if a.id == evaluation.best_approach_id)
    
    exec_prompt = ChatPromptTemplate.from_messages([
        ("system", "You solve problems thoroughly using the given approach."),
        ("human", "Solve this problem step by step.\n\nProblem: {problem}\n"
                  "Approach: {strategy}\n\nBe thorough. End with a clear answer.")
    ])
    exec_chain = exec_prompt | llm | StrOutputParser()
    
    solution = exec_chain.invoke({
        "problem": problem,
        "strategy": best.strategy,
    })
    
    return solution

In [ ]:
solution = tree_of_thought("""
A startup has $50,000/month in cloud costs and needs to cut 30%.
Breakdown: EC2 $25K, S3 $8K, RDS $10K, Data Transfer $4K, Other $3K.
What strategy should they use?
""")

print("\n" + "="*50)
print(solution)

**When to use Tree-of-Thought:**
- Strategy and design decisions
- Problems with multiple valid approaches
- Creative tasks (generate drafts, pick the best)
- When the first approach might be a dead end

**Cost:** 3-10+ API calls (generate + evaluate + execute)

**LangChain advantage:** Each stage is a separate LCEL chain. You can swap models per stage — 
e.g., use a creative model (high temp) for generation and a precise model (temp=0) for evaluation.

---

## 4. Graph-of-Thought: Non-Linear Reasoning

Trees branch but **never merge**. Graphs allow:
- **Decompose** the problem into sub-problems
- **Solve** each independently (can run in parallel!)
- **Merge** insights from multiple sub-solutions
- **Refine** the merged result if confidence is low

This is for complex problems where sub-parts interact.

In [ ]:
# ── Graph-of-Thought with Pydantic schemas ──────────────────────────

class Decomposition(BaseModel):
    sub_problems: List[str] = Field(description="Independent sub-problems")

class SubSolution(BaseModel):
    solution: str
    confidence: float = Field(ge=0.0, le=1.0)
    key_insight: str

class MergedResult(BaseModel):
    merged_solution: str
    confidence: float = Field(ge=0.0, le=1.0)

class RefinedResult(BaseModel):
    refined_solution: str
    confidence: float = Field(ge=0.0, le=1.0)


def graph_of_thought(problem: str, n_parts: int = 3) -> str:
    """Graph-of-Thought: Decompose → Solve → Merge → Refine"""

    # ── Step 1: Decompose ──
    print("Step 1: Decomposing into sub-problems...")
    decompose_chain = (
        ChatPromptTemplate.from_messages([
            ("system", "You break complex problems into independent sub-problems."),
            ("human", "Break this into {n_parts} independent sub-problems:\n\n{problem}")
        ])
        | llm.with_structured_output(Decomposition)
    )
    decomposed = decompose_chain.invoke({"n_parts": n_parts, "problem": problem})
    
    for i, sub in enumerate(decomposed.sub_problems, 1):
        print(f"  {i}. {sub[:70]}...")

    # ── Step 2: Solve each independently ──
    print("\nStep 2: Solving each sub-problem...")
    solve_chain = (
        ChatPromptTemplate.from_messages([
            ("system", "You solve sub-problems concisely and rate your confidence."),
            ("human", "Solve this concisely:\n\n{sub_problem}")
        ])
        | llm.with_structured_output(SubSolution)
    )
    
    solutions = []
    for i, sub in enumerate(decomposed.sub_problems, 1):
        sol = solve_chain.invoke({"sub_problem": sub})
        solutions.append(sol)
        print(f"  {i}. confidence={sol.confidence} | {sol.key_insight[:60]}")

    # ── Step 3: Merge ──
    print("\nStep 3: Merging insights...")
    source_text = "\n".join(
        f"Sub-solution {i+1}: {s.solution}" for i, s in enumerate(solutions)
    )
    merge_chain = (
        ChatPromptTemplate.from_messages([
            ("system", "You merge multiple sub-solutions into a unified answer."),
            ("human", "Merge these sub-solutions into one coherent answer:\n\n{solutions}")
        ])
        | llm.with_structured_output(MergedResult)
    )
    merged = merge_chain.invoke({"solutions": source_text})
    print(f"  Merged confidence: {merged.confidence}")

    # ── Step 4: Refine if confidence is low ──
    if merged.confidence < 0.85:
        print("\nStep 4: Refining (confidence was low)...")
        refine_chain = (
            ChatPromptTemplate.from_messages([
                ("system", "You improve solutions by checking consistency and fixing contradictions."),
                ("human", "Improve this solution:\n\n{solution}")
            ])
            | llm.with_structured_output(RefinedResult)
        )
        refined = refine_chain.invoke({"solution": merged.merged_solution})
        return refined.refined_solution

    return merged.merged_solution

In [ ]:
result = graph_of_thought("""
Design a notification system for a food delivery app that handles:
- Order status updates (placed, preparing, out for delivery, delivered)
- Promotional offers targeted by user preferences
- Driver communication (delayed, can't find address)
Each type has different urgency levels and delivery channels (push, SMS, email).
""")

print("\n" + "="*50)
print(result)

---
## 5. Choosing the Right Pattern

| Pattern | Structure | API Calls | LangChain Feature | Best For |
|---------|-----------|-----------|-------------------|----------|
| **CoT** | Linear chain | 1 | LCEL chain / `with_structured_output` | Math, logic, step-by-step |
| **ReAct** | Loop with tools | 2-10 | `create_react_agent` / `@tool` | External data, tool use |
| **ToT** | Branching tree | 3-10+ | Multiple LCEL chains | Strategy, design decisions |
| **GoT** | Arbitrary graph | 5-15+ | Multiple LCEL chains + Pydantic | System design, research |

**Quick decision:**
- Simple reasoning? → **CoT** with `with_structured_output()`
- Need tools or data? → **ReAct** with `create_react_agent()`
- Multiple strategies to compare? → **ToT** with separate generate/evaluate/execute chains
- Sub-problems that interact? → **GoT** with decompose/solve/merge/refine chains

---
## 6. Hands-On Exercises

### Exercise 1: Build a Structured CoT Solver with LangChain

**Goal:** Define a Pydantic schema, create an LCEL chain, and validate the output.

Complete the `TODO` sections below:

In [ ]:
# ── YOUR TASK: Complete the Pydantic schema and chain ───────────────

# TODO 1: Define the Pydantic schema for the solution
# Hint: You need a Step model and a Solution model, similar to Section 1.2
# The Solution should have: steps (list), verification (str), 
#                           final_answer (float), unit (str)

class MyStep(BaseModel):
    # TODO: Define fields — step (int), description (str), 
    #       operation (str), result (float)
    pass

class MySolution(BaseModel):
    # TODO: Define fields — steps (List[MyStep]), verification (str),
    #       final_answer (float), unit (str)
    pass


# TODO 2: Create the LCEL chain
# Hint: Use ChatPromptTemplate.from_messages() with a system + human message
#       Then pipe into llm.with_structured_output(MySolution)

my_cot_prompt = ChatPromptTemplate.from_messages([
    # TODO: Write the system message (tell the model its role and rules)
    # TODO: Write the human message with a {problem} placeholder
])

# my_cot_chain = my_cot_prompt | llm.with_structured_output(MySolution)


# TODO 3: Write the validation function
def validate_solution(solution) -> list:
    """Validate the solution. Return a list of error strings (empty = passed)."""
    errors = []
    
    # TODO: Check that steps are numbered sequentially (1, 2, 3...)
    
    # TODO: Check that all result values are numbers (float or int)
    
    # TODO: Check that final_answer matches the last step's result
    
    return errors

In [ ]:
# Test your implementation with these problems:
test_problems = [
    "A bakery makes 240 cupcakes. They sell 35% in the morning and 40% of the remaining in the afternoon. How many are left?",
    "A train travels from A to B at 80 km/h and returns at 60 km/h. Distance is 240 km. What's the average speed for the round trip?",
    "A rectangle's length is 3 times its width. Perimeter is 96 cm. Find the area.",
]

# Uncomment after completing the TODOs above:
# for i, problem in enumerate(test_problems, 1):
#     print(f"\n{'='*50}")
#     print(f"Problem {i}: {problem[:60]}...")
#     
#     solution = my_cot_chain.invoke({"problem": problem})
#     errors = validate_solution(solution)
#     
#     print(f"Answer: {solution.final_answer} {solution.unit}")
#     print(f"Steps: {len(solution.steps)}")
#     print(f"Validation: {'PASSED' if not errors else 'FAILED — ' + str(errors)}")

### Exercise 2: Build a ReAct Agent with LangGraph

**Goal:** Define custom tools with `@tool` and use `create_react_agent` to build a working agent.

Complete the `TODO` sections below:

In [ ]:
# ── City database (provided) ─────────────────────────────────────────
CITY_DB = {
    "paris": {"population": 2161000, "country": "France", "area_km2": 105.4},
    "london": {"population": 8982000, "country": "UK", "area_km2": 1572},
    "tokyo": {"population": 13960000, "country": "Japan", "area_km2": 2194},
    "new york": {"population": 8336000, "country": "USA", "area_km2": 783.8},
    "mumbai": {"population": 12442000, "country": "India", "area_km2": 603.4},
}

# TODO 1: Define tools using the @tool decorator
# Hint: The decorator reads the function name, docstring, and type hints.

@tool
def lookup_city(city_name: str) -> str:
    """Look up population, country, and area data for a city.
    Available cities: Paris, London, Tokyo, New York, Mumbai."""
    # TODO: Look up city_name in CITY_DB (case-insensitive)
    #       Return json.dumps(data) if found, or an error message if not
    pass

@tool
def calc(expression: str) -> str:
    """Evaluate a math expression. Input: a Python expression like '8982000 / 1572'."""
    # TODO: Safely evaluate the expression and return the result as a string
    pass


# TODO 2: Create the agent using LangGraph
# Hint: from langgraph.prebuilt import create_react_agent
#       agent = create_react_agent(model=llm, tools=[...], prompt="...")

# my_agent = create_react_agent(
#     model=llm,
#     tools=[lookup_city, calc],
#     prompt="TODO: Write a system prompt for a helpful city data research assistant.",
# )

In [ ]:
# TODO 3: Helper function to run the agent and print results
def ask_agent(question: str):
    """Run the agent and print the final answer."""
    # result = my_agent.invoke({"messages": [{"role": "user", "content": question}]})
    # final = result["messages"][-1].content
    # print(f"Q: {question}")
    # print(f"A: {final}")
    # return final
    pass

In [ ]:
# Test your agent with these questions (uncomment after completing TODOs):
test_questions = [
    "What is the population density of Tokyo?",
    "Which city is more densely populated: London or Mumbai?",
    "What is the combined population of all European cities in the database?",
]

# for q in test_questions:
#     print(f"\n{'='*50}")
#     ask_agent(q)

---

## Recap

Today you implemented four structured prompting patterns **using LangChain**:

1. **Structured CoT** — LCEL chains (`prompt | model | parser`) for clean reasoning pipelines
2. **Programmatic CoT** — `with_structured_output()` + Pydantic for validated, typed responses
3. **ReAct** — `@tool` decorator + `create_react_agent()` from LangGraph for production agents
4. **Tree/Graph-of-Thought** — Composable LCEL chains with different models per stage

**LangChain patterns you learned:**

| Pattern | Code |
|---------|------|
| LCEL chain | `prompt \| llm \| StrOutputParser()` |
| Structured output | `llm.with_structured_output(MyPydanticModel)` |
| Tool definition | `@tool` decorator with type hints + docstring |
| Tool binding | `llm.bind_tools([tool1, tool2])` |
| ReAct agent | `create_react_agent(model=llm, tools=tools)` |
| Prompt template | `ChatPromptTemplate.from_messages([("system", "..."), ("human", "{var}")])` |

**Packages used:** `langchain-openai`, `langchain-core`, `langgraph`, `pydantic`

**Next session:** Self-Reflection & Critique — agents that evaluate and improve their own outputs.

---
*© BIA® School of Technology & AI — Generative AI & Agentic AI Development Program*